# 🤖 Machine Learning — Notebook de Projeto

**Disciplina:** Machine Learning  
**Professor:** Messias Batista  
**Aluno(a):**  
**Data:**  
**Dataset:** Amazon Product Reviews (amazon.csv)  
**Problema de negócio:** Classificar avaliações de produtos Amazon em 5 categorias de rating (1–5 estrelas) utilizando técnicas de NLP combinadas com redes neurais (MLP sklearn, MLP Keras, LSTM e CNN).  

---
## 1. 📦 Importações

Importe aqui todas as bibliotecas que serão utilizadas ao longo do projeto.

Você precisará de bibliotecas para:
- **Manipulação de dados** — leitura, transformação e análise de tabelas
- **Visualização** — criação de gráficos e figuras
- **Pré-processamento** — divisão dos dados, normalização e codificação de variáveis
- **Algoritmos de Machine Learning** — os modelos que serão treinados
- **Métricas de avaliação** — para medir o desempenho dos modelos

In [ ]:
!pip install keras.utils
!pip install np_utils

import re
import itertools
import random
import sys
import os
from abc import ABCMeta
from collections import defaultdict

import numpy as np
import pandas as pd
import tensorflow as tf
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import cm
%matplotlib inline
plt.style.use('ggplot')

# NLP
import nltk
from bs4 import BeautifulSoup
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
english_stemmer = nltk.stem.SnowballStemmer('english')

# Scipy
import six
from scipy import sparse
from scipy.sparse import csr_matrix, issparse

# Sklearn
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.feature_selection import SelectKBest, chi2, f_classif
from sklearn.preprocessing import normalize, binarize, LabelBinarizer
from sklearn.linear_model import SGDClassifier, SGDRegressor
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils import check_X_y, check_array
from sklearn.utils.extmath import safe_sparse_dot
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Keras / TensorFlow
from keras.preprocessing import sequence
from tensorflow.keras.utils import to_categorical
from keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Lambda
from keras.layers import Embedding
from keras.layers import LSTM, SimpleRNN, GRU
from keras.layers import Convolution1D
from tensorflow.keras.preprocessing.text import Tokenizer
from keras import backend as K

---
## 2. 📂 Carregamento dos Dados

Carregue o dataset e faça uma primeira inspeção para entender com o que você está trabalhando.

Nesta etapa você deve responder:
- Quantas linhas e colunas o dataset possui?
- Quais são os tipos de cada coluna?
- Existem valores nulos? Em quais colunas e em qual quantidade?

In [ ]:
# Faça upload do arquivo amazon.csv no Google Colab antes de executar
# Link: https://drive.google.com/open?id=1JmGqiLEh_wUeLilLF7Hv_w4jgUNbG0Q_
data = pd.read_csv('/content/amazon.csv')

# Limitando a 1000 registros para fins didáticos
data = data[:1000]
data.head()

In [ ]:
print(f"Dimensões: {data.shape}")
print(f"\nColunas: {list(data.columns)}")
print(f"\nTipos de dados:\n{data.dtypes}")
print(f"\nValores nulos por coluna:\n{data.isnull().sum()}")

---
## 3. 🔍 Análise Exploratória de Dados (EDA)

Explore os dados antes de construir qualquer modelo. Esta é uma das etapas mais importantes do processo.

Nesta etapa você deve:
- Calcular estatísticas descritivas das variáveis numéricas
- Analisar a distribuição da variável alvo (está balanceada?)
- Visualizar a distribuição das demais variáveis
- Identificar possíveis outliers
- Analisar a correlação entre as variáveis

In [ ]:
# Estatísticas descritivas
data.describe()

In [ ]:
# Distribuição da variável alvo (Rating)
print(data['Rating'].value_counts().sort_index())

plt.figure(figsize=(7, 4))
sns.countplot(x=data['Rating'])
plt.title('Distribuição dos Ratings')
plt.xlabel('Rating (estrelas)')
plt.ylabel('Quantidade')
plt.tight_layout()
plt.show()

In [ ]:
# Comprimento das reviews (número de palavras)
data['review_length'] = data['Reviews'].dropna().apply(lambda x: len(str(x).split()))
plt.figure(figsize=(10, 4))
data['review_length'].hist(bins=30)
plt.title('Comprimento das Reviews (número de palavras)')
plt.xlabel('Número de Palavras')
plt.ylabel('Frequência')
plt.tight_layout()
plt.show()

# Exemplos de reviews por rating
print("\nExemplos de reviews por rating:")
for rating in [1, 3, 5]:
    subset = data[data['Rating'] == rating]['Reviews'].dropna()
    if len(subset) > 0:
        print(f"\nRating {rating}: {subset.iloc[0][:200]}...")

---
## 4. 🛠️ Pré-processamento

Prepare os dados para que o modelo consiga aprender corretamente.

Nesta etapa você deve:
- Remover colunas que não contribuem para o modelo
- Tratar os valores nulos (remover ou preencher)
- Converter variáveis categóricas em numéricas
- Separar as features (X) da variável alvo (y)
- Aplicar normalização ou padronização se necessário

In [ ]:
# Remover reviews nulas
data = data[data['Reviews'].isnull() == False]
print(f"Registros após remoção de nulos: {data.shape[0]}")

def review_to_wordlist(review, remove_stopwords=True):
    """Converte uma review em lista de palavras limpas com stemming.

    Etapas:
    1. Remove HTML
    2. Remove caracteres não-alfabéticos
    3. Converte para minúsculas
    4. Remove stop words (opcional)
    5. Aplica stemming
    """
    review_text = BeautifulSoup(review, 'html.parser').get_text()
    review_text = re.sub("[^a-zA-Z]", " ", review)
    words = review_text.lower().split()
    if remove_stopwords:
        stops = set(stopwords.words("english"))
        words = [w for w in words if w not in stops]
    b = []
    stemmer = english_stemmer
    for word in words:
        b.append(stemmer.stem(word))
    return b

nltk.download('stopwords')

In [ ]:
# Separação de X (textos) e y (rating) — vetorização ocorre após o split
x_raw = data['Reviews']
y_raw = data['Rating']

print(f"Total de amostras: {len(x_raw)}")
print(f"Classes (ratings): {sorted(y_raw.unique())}")

---
## 5. ✂️ Separação Treino / Teste

Divida os dados em dois conjuntos: um para treinar o modelo e outro para testá-lo.

Lembre-se:
- O modelo deve ser treinado **apenas** com os dados de treino
- Os dados de teste simulam situações novas, que o modelo nunca viu
- Em NLP, o split deve ocorrer **antes** da vetorização TF-IDF para evitar data leakage
- O vetorizador deve ser ajustado (`fit`) apenas nos dados de treino

In [ ]:
# Divisão treino/teste ANTES da vetorização (evita data leakage)
train, test = train_test_split(data, test_size=0.3, random_state=42)

# Pré-processamento textual aplicado separadamente em treino e teste
clean_train_reviews = []
for review in train['Reviews']:
    clean_train_reviews.append(" ".join(review_to_wordlist(review)))

clean_test_reviews = []
for review in test['Reviews']:
    clean_test_reviews.append(" ".join(review_to_wordlist(review)))

# TF-IDF para modelos clássicos (fit apenas no treino)
vectorizer = TfidfVectorizer(
    min_df=2, max_df=0.95, max_features=200000, ngram_range=(1, 4), sublinear_tf=True
)
vectorizer = vectorizer.fit(clean_train_reviews)
train_features = vectorizer.transform(clean_train_reviews)
test_features  = vectorizer.transform(clean_test_reviews)

# Seleção das k=100 melhores features (fit apenas no treino)
fselect = SelectKBest(chi2, k=100)
train_features = fselect.fit_transform(train_features, train['Rating'])
test_features  = fselect.transform(test_features)

print(f"Treino: {len(clean_train_reviews)} amostras | Features: {train_features.shape[1]}")
print(f"Teste:  {len(clean_test_reviews)}  amostras | Features: {test_features.shape[1]}")

---
## 6. 🧠 Treinamento do Modelo

Escolha um algoritmo, instancie o modelo e treine-o com os dados de treino.

Lembre-se:
- O treinamento acontece com o método `fit()`
- As previsões são feitas com o método `predict()`
- Você pode testar mais de um algoritmo e compará-los na seção 8

In [ ]:
# Vetorização específica para redes neurais (max_features menor para convergência)
vectorizer_mlp = TfidfVectorizer(
    min_df=2, max_df=0.95, max_features=1000, ngram_range=(1, 3), sublinear_tf=True
)
vectorizer_mlp = vectorizer_mlp.fit(clean_train_reviews)
train_features_mlp = vectorizer_mlp.transform(clean_train_reviews)
test_features_mlp  = vectorizer_mlp.transform(clean_test_reviews)

batch_size = 32
nb_classes = 5

X_train = train_features_mlp.toarray()
X_test  = test_features_mlp.toarray()
y_train = np.array(train['Rating'] - 1)
y_test  = np.array(test['Rating']  - 1)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')

# Modelo principal: MLP Classifier (sklearn)
clf = MLPClassifier(
    solver='lbfgs', alpha=1e-5, hidden_layer_sizes=(15, 30), random_state=1
)
clf.fit(X_train, y_train)
print("\nMLP (sklearn) treinado com sucesso.")

In [ ]:
# Previsões do MLP sklearn
y_prev = clf.predict(X_test)
print(f"predição MLP (SKLearn) - accuracy: {accuracy_score(y_test, y_prev):.4f}")

---
## 7. 📊 Avaliação do Modelo

Meça o desempenho do modelo utilizando métricas adequadas ao problema.

Para problemas de **classificação**, avalie:
- **Acurácia** — proporção de acertos em relação ao total
- **Matriz de Confusão** — visualização detalhada dos acertos e erros por classe
- **Relatório de Classificação** — precision, recall e f1-score por classe
- **Validação Cruzada** — para uma estimativa mais robusta e confiável do desempenho

In [ ]:
# Acurácia
print(f"Acurácia (MLP sklearn): {accuracy_score(y_test, y_prev):.4f}")

In [ ]:
# Matriz de Confusão
def plot_confusion_matrix(cm, classes, normalize=False,
                          title='Matriz de confusão', cmap=plt.cm.Blues):
    """Imprime e plota a matriz de confusão.
    Normalização pode ser aplicada configurando normalize=True.
    """
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Matriz de confusão normalizada")
    else:
        print('Matriz de confusão, não normalizada')
    print(cm)
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, cm[i, j], horizontalalignment='center',
                 color='white' if cm[i, j] > thresh else 'black')
    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.show()

cnf_matrix = confusion_matrix(y_test, y_prev)
plot_confusion_matrix(cnf_matrix, classes=['1','2','3','4','5'],
                      title='Matriz de Confusão — MLP sklearn')

In [ ]:
# Relatório de Classificação
print(classification_report(y_test, y_prev, target_names=['1','2','3','4','5']))

In [ ]:
# Validação Cruzada via Pipeline (evita leakage no TF-IDF)
pipeline_mlp = Pipeline([
    ('tfidf', TfidfVectorizer(min_df=2, max_df=0.95, max_features=1000,
                              ngram_range=(1, 3), sublinear_tf=True)),
    ('mlp',   MLPClassifier(solver='lbfgs', alpha=1e-5,
                             hidden_layer_sizes=(15, 30), random_state=1)),
])
all_clean = [" ".join(review_to_wordlist(r)) for r in data['Reviews']]
cv_scores = cross_val_score(pipeline_mlp, all_clean, data['Rating'] - 1,
                            scoring='accuracy', cv=3)
print(f"Cross-validation (3 folds): {cv_scores}")
print(f"Média: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

---
## 8. 🏆 Comparação de Modelos

Teste outros algoritmos e compare os resultados para identificar o mais adequado ao seu problema.

Dica: use validação cruzada para comparar os modelos de forma justa,
pois ela elimina o efeito da aleatoriedade de uma única divisão treino/teste.

In [ ]:
# ============================================================
# Modelos Clássicos de ML (TF-IDF 200k + SelectKBest k=100)
# ============================================================
model1 = MultinomialNB(alpha=0.001)
model1.fit(train_features, train['Rating'])

model2 = SGDClassifier(loss='modified_huber', random_state=0, shuffle=True)
model2.fit(train_features, train['Rating'])

model3 = RandomForestClassifier(random_state=42)
model3.fit(train_features, train['Rating'])

model4 = GradientBoostingClassifier(random_state=42)
model4.fit(train_features, train['Rating'])

pred_1 = model1.predict(test_features.toarray())
pred_2 = model2.predict(test_features.toarray())
pred_3 = model3.predict(test_features.toarray())
pred_4 = model4.predict(test_features.toarray())

# Relatório detalhado do SGD
print("Relatório de Classificação — SGD Classifier:")
print(classification_report(test['Rating'], pred_2, target_names=['1','2','3','4','5']))

# ============================================================
# NBSVM — Naive Bayes + SVM (implementação customizada)
# ============================================================
class NBSVM(six.with_metaclass(ABCMeta, BaseEstimator, ClassifierMixin)):

    def __init__(self, alpha=1.0, C=1.0, max_iter=10000):
        self.alpha = alpha
        self.max_iter = max_iter
        self.C = C
        self.svm_ = []

    def fit(self, X, y):
        X, y = check_X_y(X, y, 'csr')
        _, n_features = X.shape
        labelbin = LabelBinarizer()
        Y = labelbin.fit_transform(y)
        self.classes_ = labelbin.classes_
        if Y.shape[1] == 1:
            Y = np.concatenate((1 - Y, Y), axis=1)
        Y = Y.astype(np.float64)
        n_effective_classes = Y.shape[1]
        self.class_count_ = np.zeros(n_effective_classes, dtype=np.float64)
        self.ratios_ = np.full((n_effective_classes, n_features), self.alpha, dtype=np.float64)
        self._compute_ratios(X, Y)
        for i in range(n_effective_classes):
            X_i = X.multiply(self.ratios_[i])
            svm = LinearSVC(C=self.C, max_iter=self.max_iter)
            Y_i = Y[:, i]
            svm.fit(X_i, Y_i)
            self.svm_.append(svm)
        return self

    def predict(self, X):
        n_effective_classes = self.class_count_.shape[0]
        n_examples = X.shape[0]
        D = np.zeros((n_effective_classes, n_examples))
        for i in range(n_effective_classes):
            X_i = X.multiply(self.ratios_[i])
            D[i] = self.svm_[i].decision_function(X_i)
        return self.classes_[np.argmax(D, axis=0)]

    def _compute_ratios(self, X, Y):
        '''Contar ocorrências de recursos e proporções de computação.'''
        if np.any((X.data if issparse(X) else X) < 0):
            raise ValueError("Entrada X deveria ser positiva")
        self.ratios_ += safe_sparse_dot(Y.T, X)
        normalize(self.ratios_, norm='l1', axis=1, copy=False)
        row_calc = lambda r: np.log(np.divide(r, (1 - r)))
        self.ratios_ = np.apply_along_axis(row_calc, axis=1, arr=self.ratios_)
        check_array(self.ratios_)
        self.ratios_ = sparse.csr_matrix(self.ratios_)


def f1_class(pred, truth, class_val):
    n = len(truth)
    truth_class = pred_class = tp = 0
    for ii in range(n):
        if truth[ii] == class_val:
            truth_class += 1
            if truth[ii] == pred[ii]:
                tp += 1
                pred_class += 1
                continue
        if pred[ii] == class_val:
            pred_class += 1
    precision = tp / float(pred_class)
    recall    = tp / float(truth_class)
    return (2.0 * precision * recall) / (precision + recall)


def semeval_senti_f1(pred, truth, pos=2, neg=0):
    return (f1_class(pred, truth, pos) + f1_class(pred, truth, neg)) / 2.0


model5 = NBSVM(C=0.01)
model5.fit(train_features, train['Rating'])
pred_5 = model5.predict(test_features)

# Matriz de confusão do NBSVM
cnf_matrix_5 = confusion_matrix(test['Rating'], pred_5)
plot_confusion_matrix(cnf_matrix_5, classes=['1','2','3','4','5'],
                      title='Matriz de Confusão — NBSVM')

# ============================================================
# MLP com Keras (mesma vetorização TF-IDF 1000)
# ============================================================
X_train_k = X_train.copy().astype('float32')
X_test_k  = X_test.copy().astype('float32')

Y_train = to_categorical(y_train, nb_classes)
Y_test  = to_categorical(y_test,  nb_classes)

scale = np.max(X_train_k)
X_train_k /= scale
X_test_k  /= scale
mean = np.mean(X_train_k)
X_train_k -= mean
X_test_k  -= mean

input_dim = X_train_k.shape[1]

model_mlp_keras = Sequential()
model_mlp_keras.add(Dense(256, input_dim=input_dim))
model_mlp_keras.add(Activation('relu'))
model_mlp_keras.add(Dropout(0.4))
model_mlp_keras.add(Dense(128))
model_mlp_keras.add(Activation('relu'))
model_mlp_keras.add(Dropout(0.2))
model_mlp_keras.add(Dense(nb_classes))
model_mlp_keras.add(Activation('softmax'))
model_mlp_keras.compile(loss='categorical_crossentropy', optimizer='rmsprop')
model_mlp_keras.fit(X_train_k, Y_train, epochs=5, batch_size=16, validation_split=0.1)

preds_mlp_keras = np.argmax(model_mlp_keras.predict(X_test_k), axis=1)

# ============================================================
# LSTM
# ============================================================
max_features   = 20000
EMBEDDING_DIM  = 100
VALIDATION_SPLIT = 0.2
maxlen         = 70

tokenizer = Tokenizer(num_words=max_features)
tokenizer.fit_on_texts(train['Reviews'])
sequences_train = tokenizer.texts_to_sequences(train['Reviews'])
sequences_test  = tokenizer.texts_to_sequences(test['Reviews'])

print('Pad sequences (samples x time)')
X_train_seq = sequence.pad_sequences(sequences_train, maxlen=maxlen)
X_test_seq  = sequence.pad_sequences(sequences_test,  maxlen=maxlen)
print(f'X_train shape: {X_train_seq.shape}')
print(f'X_test shape:  {X_test_seq.shape}')

model_lstm = Sequential()
model_lstm.add(Embedding(max_features, 128))
model_lstm.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model_lstm.add(Dense(nb_classes))
model_lstm.add(Activation('softmax'))
model_lstm.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

print('Treinamento LSTM...')
model_lstm.fit(X_train_seq, Y_train, batch_size=batch_size, epochs=1,
               validation_data=(X_test_seq, Y_test))
score, acc = model_lstm.evaluate(X_test_seq, Y_test, batch_size=batch_size)
print(f'LSTM — Test score: {score:.4f} | Test accuracy: {acc:.4f}')
preds_lstm = np.argmax(model_lstm.predict(X_test_seq), axis=1)

# ============================================================
# CNN (ConvNet)
# ============================================================
nb_filter    = 32
filter_length = 3
hidden_dims  = 250
nb_epoch     = 2

def max_1d(X):
    return tf.reduce_max(X, axis=1)

model_cnn = Sequential()
model_cnn.add(Embedding(max_features, 128))
model_cnn.add(Convolution1D(nb_filter, filter_length, padding='valid', activation='relu'))
model_cnn.add(Lambda(max_1d, output_shape=(nb_filter,)))
model_cnn.add(Dense(hidden_dims))
model_cnn.add(Dropout(0.2))
model_cnn.add(Activation('relu'))
model_cnn.add(Dense(nb_classes))
model_cnn.add(Activation('sigmoid'))
model_cnn.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

print('Treinamento CNN...')
model_cnn.fit(X_train_seq, Y_train, batch_size=batch_size, epochs=1,
              validation_data=(X_test_seq, Y_test))
score, acc = model_cnn.evaluate(X_test_seq, Y_test, batch_size=batch_size)
print(f'CNN — Test score: {score:.4f} | Test accuracy: {acc:.4f}')
preds_cnn = np.argmax(model_cnn.predict(X_test_seq), axis=1)

# ============================================================
# Tabela Comparativa de Acurácias
# ============================================================
print("\n" + "=" * 48)
print(f"{'Modelo':<32} {'Acurácia':>10}")
print("=" * 48)
print(f"{'predição 1 - Naive Bayes':<32} {accuracy_score(test['Rating'], pred_1):>10.4f}")
print(f"{'predição 2 - SGD Classifier':<32} {accuracy_score(test['Rating'], pred_2):>10.4f}")
print(f"{'predição 3 - Random Forest':<32} {accuracy_score(test['Rating'], pred_3):>10.4f}")
print(f"{'predição 4 - Gradient Boosting':<32} {accuracy_score(test['Rating'], pred_4):>10.4f}")
print(f"{'predição 5 - NBSVM':<32} {accuracy_score(test['Rating'], pred_5):>10.4f}")
print(f"{'predição 6 - MLP (sklearn)':<32} {accuracy_score(y_test, y_prev):>10.4f}")
print(f"{'predição 7 - MLP (Keras)':<32} {accuracy_score(y_test, preds_mlp_keras):>10.4f}")
print(f"{'predição 8 - LSTM':<32} {accuracy_score(test['Rating'], preds_lstm + 1):>10.4f}")
print(f"{'predição 9 - CNN':<32} {accuracy_score(test['Rating'], preds_cnn + 1):>10.4f}")
print("=" * 48)

---
## 9. 📝 Conclusões

Responda as perguntas abaixo com base nos resultados obtidos:

**1. Qual algoritmo apresentou melhor desempenho? Por quê?**  
> _Escreva aqui_

**2. O modelo está sofrendo overfitting ou underfitting? Como você identificou?**  
> _Escreva aqui_

**3. Os resultados respondem à pergunta de negócio levantada no início?**  
> _Escreva aqui_

**4. Qual seria o próximo passo para melhorar o modelo?**  
> _Escreva aqui_

---
### 🔖 Referências
- Dataset: Amazon Product Reviews — https://drive.google.com/open?id=1JmGqiLEh_wUeLilLF7Hv_w4jgUNbG0Q_
- Scikit-learn MLPClassifier: https://scikit-learn.org/stable/modules/neural_networks_supervised.html
- TensorFlow/Keras: https://www.tensorflow.org/api_docs/python/tf/keras
- Material da disciplina: www.mrafaelbatista.dev
